# M11 — Dueling DQL: Full 6000-Episode Training Run (Colab T4)

This notebook trains the **Dueling DQL (M11)** baseline for the full paper-scale budget (**6000 episodes**, matching PKTD3-TD's Table III `M_EPISODES`), producing a fair benchmark comparison for M14.

**Architecture & Training:**
- 200-action discrete velocity grid (5 speeds × 5 polar angles × 8 azimuth angles)
- Dueling Q-Network (Value stream + Advantage stream with mean-subtraction identifiability)
- Epsilon-greedy exploration linearly decaying from 1.0 to 0.05
- Polyak soft target updates (tau=0.005) executed every gradient step

**Workflow pattern:**
1. Cloned fresh to local Colab SSD (`/content/uav_trajectory_rl`) for fast I/O.
2. Checkpoints saved directly to **Google Drive** for persistent safety against disconnects.
3. At the end, 30-seed deterministic evaluation is run, checkpoints copied to repo, and pushed to GitHub using Colab's `GITHUB_PAT_TOKEN` secret.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone the repo fresh (local disk, not Drive) and install

In [2]:
# Clone repo fresh to local disk with authenticated URL for seamless push at the end
import os
from google.colab import userdata

try:
    github_token = userdata.get('GITHUB_PAT_TOKEN')
except Exception:
    github_token = None

repo_owner = "Krishna200608"
repo_name = "uav_trajectory_rl"

if github_token:
    repo_url = f"https://{github_token}@github.com/{repo_owner}/{repo_name}.git"
    print("Authenticated git URL configured using GITHUB_PAT_TOKEN secret.")
else:
    repo_url = f"https://github.com/{repo_owner}/{repo_name}.git"
    print("WARNING: GITHUB_PAT_TOKEN secret not found. Git push in Cell 8 will require manual auth.")

%cd /content
!rm -rf uav_trajectory_rl
!git clone {repo_url} uav_trajectory_rl
%cd uav_trajectory_rl
!pip install -e . --quiet
!git log --oneline -n 5

Authenticated git URL configured using GITHUB_PAT_TOKEN secret.
/content
Cloning into 'uav_trajectory_rl'...
remote: Enumerating objects: 553, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 553 (delta 36), reused 67 (delta 28), pack-reused 471 (from 1)
Receiving objects: 100% (553/553), 177.80 MiB | 19.90 MiB/s, done.
Resolving deltas: 100% (303/303), done.
Updating files: 100% (133/133), done.
/content/uav_trajectory_rl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for uav-trajectory-rl (pyproject.toml) ... done
de043b2 (HEAD -> main, origin/main, origin/HEAD) Update README.md with Dueling DQL and PPO Colab training notebooks
a2fff2a Add and calibrate Colab training notebooks for Dueling DQL and PPO baselines
b309457 Document Greedy's term

## 3. Confirm GPU and check the training script's actual CLI flags

**Important:** run this before launching the long job. Confirm the exact flag names/units (episodes vs. total steps, checkpoint-every units) match what's assumed below — if `train_dueling_dql.py`'s help text differs, adjust Cell 6 accordingly rather than assuming.

In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print()
!python scripts/train_dueling_dql.py --help

CUDA available: True
Device: Tesla T4

usage: train_dueling_dql.py [-h] [--episodes EPISODES] [--k K]
                            [--batch-size BATCH_SIZE] [--seed SEED]
                            [--checkpoint-dir CHECKPOINT_DIR]
                            [--checkpoint-every CHECKPOINT_EVERY]
                            [--log-every LOG_EVERY] [--lr LR] [--gamma GAMMA]
                            [--tau TAU] [--epsilon-start EPSILON_START]
                            [--epsilon-end EPSILON_END]
                            [--epsilon-decay-episodes EPSILON_DECAY_EPISODES]
                            [--replay-size REPLAY_SIZE] [--no-progress-bar]
                            [--resume] [--resume-from RESUME_FROM]

Train Dueling DQL Baseline for 3D UAV Trajectory Design

options:
  -h, --help            show this help message and exit
  --episodes EPISODES   Total training episodes (default: 6000)
  --k K                 Number of ground users (default: 10)
  --batch-size BATCH_SIZE
 

## 4. Set the Drive checkpoint path (persistent across disconnects)

In [4]:
import os
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/dueling_dql_run1"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be written to:', DRIVE_CHECKPOINT_DIR)

Checkpoints will be written to: /content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/dueling_dql_run1


## 5. Launch the full 6000-episode training run

Matches PKTD3-TD's episode budget (Table III `M_EPISODES=6000`) for a fair comparison. `--checkpoint-every 250` gives 24 checkpoints across the run — same cadence used for `run4`, which worked well for monitoring.

**This cell will run for a while.** Since Dueling DQL is a single Q-network (no twin critic, no actor network), it should train noticeably faster per-episode than PKTD3-TD did — but let it run to completion; don't interrupt based on early impressions.

In [5]:
!python scripts/train_dueling_dql.py \
  --episodes 6000 \
  --seed 0 \
  --checkpoint-dir "{DRIVE_CHECKPOINT_DIR}" \
  --checkpoint-every 250 \
  --log-every 50

STARTING DUELING DQL BASELINE TRAINING (M11)
Episodes: 6000 | K: 10 | Batch: 128 | Seed: 0
Actions: 200 | Epsilon: 1.0 -> 0.05 (over 4800 eps)
Checkpoint Dir: /content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/dueling_dql_run1
Episode   50/6000 | reward=+122.594 | avg(50)=+167.917 | eps=0.990 | buf=10000 | updates=9873
Episode  100/6000 | reward=+123.409 | avg(50)=+190.833 | eps=0.980 | buf=20000 | updates=19873
Episode  150/6000 | reward=+423.333 | avg(50)=+208.887 | eps=0.970 | buf=30000 | updates=29873
Episode  200/6000 | reward=+583.931 | avg(50)=+236.976 | eps=0.960 | buf=40000 | updates=39873
Episode  250/6000 | reward=+377.122 | avg(50)=+293.727 | eps=0.951 | buf=50000 | updates=49873
Episode  300/6000 | reward=+395.361 | avg(50)=+333.688 | eps=0.941 | buf=60000 | updates=59873
Episode  350/6000 | reward=+539.611 | avg(50)=+343.380 | eps=0.931 | buf=70000 | updates=69873
Episode  400/6000 | reward=+446.911 | avg(50)=+361.125 | eps=0.921 | buf=80000 | updates=79873
Epi

## 6. Sanity-check the run completed properly

In [6]:
import glob, os, numpy as np

ckpts = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*.pt"))
print(f'Found {len(ckpts)} checkpoint files in Drive:')
for c in ckpts[-5:]:
    print(' ', os.path.basename(c))

rewards_path = f"{DRIVE_CHECKPOINT_DIR}/episode_rewards.npy"
if os.path.exists(rewards_path):
    r = np.load(rewards_path)
    print(f'\nTotal episodes logged: {len(r)}')
    print(f'First 100 mean reward: {r[:100].mean():.2f}')
    print(f'Last 100 mean reward:  {r[-100:].mean():.2f}')
    print(f'Max episode reward:    {r.max():.2f}')
else:
    print('WARNING: episode_rewards.npy not found -- check the run completed correctly.')

Found 25 checkpoint files in Drive:
  dueling_dql_ep5500.pt
  dueling_dql_ep5750.pt
  dueling_dql_ep6000.pt
  dueling_dql_ep750.pt
  dueling_dql_final.pt

Total episodes logged: 6000
First 100 mean reward: 179.37
Last 100 mean reward:  1482.55
Max episode reward:    1821.21


## 7. Behavioral check — arrival rate, not just reward

Learned from the PKTD3-TD investigation: never judge a run by reward alone. Run this before deciding the run is usable.

In [8]:
import glob, os, sys, torch
import numpy as np

# 1. Clean out any poisoned or incomplete module cache from sys.modules
for mod in list(sys.modules.keys()):
    if mod == 'uav_trajectory_rl' or mod.startswith('uav_trajectory_rl.'):
        del sys.modules[mod]

# 2. Dynamically locate the actual src/ directory containing uav_trajectory_rl
src_candidates = [
    '/content/uav_trajectory_rl/src',
    os.path.abspath('src'),
    os.path.abspath('.'),
    '/content/src',
    '/content/drive/MyDrive/uav_trajectory_rl/src',
    '/content/drive/MyDrive/Uav_trajectory_rl/src',
]

src_path = None
for p_dir in src_candidates:
    if os.path.exists(os.path.join(p_dir, 'uav_trajectory_rl', 'mdp_environment.py')):
        src_path = p_dir
        break

if not src_path:
    matches = glob.glob('/content/**/uav_trajectory_rl/mdp_environment.py', recursive=True)
    if matches:
        src_path = os.path.dirname(os.path.dirname(matches[0]))

if src_path:
    print('Package source located at:', src_path)
    while src_path in sys.path:
        sys.path.remove(src_path)
    sys.path.insert(0, src_path)
else:
    raise FileNotFoundError('Could not locate uav_trajectory_rl package source on disk. Please confirm Step 2 cloned successfully.')

from uav_trajectory_rl.mdp_environment import UAVTrajectoryEnv
from uav_trajectory_rl.baselines.dueling_dql import DuelingDQLAgent, discrete_action_to_physical

# Locate final checkpoint
final_ckpt_candidates = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*final*.pt")) or sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*.pt"))
if not final_ckpt_candidates:
    raise FileNotFoundError(f"No checkpoint files found in {DRIVE_CHECKPOINT_DIR}")
final_ckpt = final_ckpt_candidates[-1]
print('Evaluating checkpoint:', final_ckpt)

env_tmp = UAVTrajectoryEnv(k=10, rng=np.random.default_rng(0))
agent = DuelingDQLAgent(state_dim=env_tmp.state_dim, num_actions=200)
agent.load(final_ckpt)
agent.q_net.eval()
agent.q_target.eval()

def rollout(seed, k=10):
    env = UAVTrajectoryEnv(k=k, rng=np.random.default_rng(seed))
    state = env.reset()
    start = env.uav_pos.copy()
    max_dist, done, steps, ep_reward, arrived = 0.0, False, 0, 0.0, False
    while not done and steps < 200:
        a_idx = agent.select_action(state, epsilon=0.0)  # deterministic eval
        v, lam, rho = discrete_action_to_physical(a_idx)
        state, r, done, info = env.step((v, lam, rho))
        max_dist = max(max_dist, float(np.linalg.norm(env.uav_pos - start)))
        ep_reward += r
        steps += 1
        arrived = info.get('arrived', False)
    return max_dist, ep_reward, arrived, steps

dists, rewards, arrivals, steps_taken = [], [], [], []
for seed in range(30):
    d, r, a, s = rollout(seed)
    dists.append(d); rewards.append(r); arrivals.append(a); steps_taken.append(s)
dists = np.array(dists)

print(f'\n30-seed evaluation (deterministic, epsilon=0.0):')
print(f'  Mean max displacement:   {dists.mean():.1f} m')
print(f'  Median max displacement: {np.median(dists):.1f} m')
print(f'  Frac > 50m:              {(dists > 50).mean():.1%}')
print(f'  ARRIVAL RATE:            {np.mean(arrivals):.1%} ({sum(arrivals)}/30)')
print(f'  Mean episode reward:     {np.mean(rewards):.2f}')
print(f'  Mean steps taken:        {np.mean(steps_taken):.1f}')


Package source located at: /content/uav_trajectory_rl/src
Evaluating checkpoint: /content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/dueling_dql_run1/dueling_dql_final.pt

30-seed evaluation (deterministic, epsilon=0.0):
  Mean max displacement:   600.2 m
  Median max displacement: 602.8 m
  Frac > 50m:              100.0%
  ARRIVAL RATE:            0.0% (0/30)
  Mean episode reward:     1320.39
  Mean steps taken:        200.0


## 8. Copy results into the local repo clone and push

**Run this only after confirming Cells 6-7 look reasonable.**

In [9]:
import shutil

LOCAL_CHECKPOINT_DIR = "/content/uav_trajectory_rl/checkpoints/dueling_dql_run1"
os.makedirs(LOCAL_CHECKPOINT_DIR, exist_ok=True)

for f in glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*"):
    shutil.copy(f, LOCAL_CHECKPOINT_DIR)

print('Copied files:')
!ls -la "{LOCAL_CHECKPOINT_DIR}"

Copied files:
total 49868
drwxr-xr-x 2 root root    4096 Aug 31 08:55 .
drwxr-xr-x 7 root root    4096 Aug 31 08:55 ..
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep1000.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep1250.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep1500.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep1750.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep2000.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep2250.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep2500.pt
-rw------- 1 root root 2002679 Aug 31 08:55 dueling_dql_ep250.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep2750.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep3000.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep3250.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep3500.pt
-rw------- 1 root root 2002725 Aug 31 08:55 dueling_dql_ep3750.pt
-rw------- 1 root root 2

In [10]:
%cd /content/uav_trajectory_rl
!git config user.email "krishnasikheriya001@gmail.com"
!git config user.name "Krishna200608"

# Ensure remote URL has the token for non-interactive push
import os
if github_token:
    os.system(f"git remote set-url origin {repo_url}")

!git add -f checkpoints/dueling_dql_run1
!git status
!git commit -m "Add Dueling DQL (M11) full training run and checkpoints"

# Pull latest remote commits with rebase to avoid push rejection
!git pull --rebase origin main
!git push origin main
!git log --oneline -n 3


/content/uav_trajectory_rl
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep1000.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep1250.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep1500.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep1750.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep2000.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep2250.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep250.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep2500.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep2750.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep3000.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep3250.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep3500.pt
	new file:   checkpoints/dueling_dql_run1/dueling_dql_ep3750.pt
	ne

## Done

Bring the results (Step 7's evaluation numbers, and the pushed commit hash) back to the main conversation for independent review before this is used in M14's comparison plots.